# PMXT data-adapter pilot

A separate development-only audit of PMXT v2 historical depth for supported negRisk and ladder baskets. The Dome data, fitted models and completed final evaluation stay unchanged. This notebook does **not** train models or estimate trading P&L.

See [the pilot report](../docs/reports/pmxt_pilot.md). The default cells verify and display saved pilot results without network requests. The planned week and actually cached/replayed interval are displayed separately.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))
from src.data.pmxt_pilot import DEFAULT_ROOT, discover, download, verify
from src.data.pmxt_audit import audit
from src.common import sha256
ROOT = DEFAULT_ROOT
RESULTS = PROJECT / 'results' / 'pmxt_pilot' / ROOT.name

# Opt-in only. The shipped notebook makes no HTTP requests.
RUN_DISCOVERY = False
DOWNLOAD_HOURS = 0  # 2 = warmup + one analysis hour; 169 = planned full week
TRANSFER_GB = 2    # bounded logical range bytes per invocation
REBUILD_AUDIT = False
VERIFY_DOME = True
if RUN_DISCOVERY:
    discover(ROOT)
if DOWNLOAD_HOURS:
    download(ROOT, hours=DOWNLOAD_HOURS, transfer_gb=TRANSFER_GB)
if REBUILD_AUDIT:
    audit(ROOT)

In [2]:
summary = json.loads((RESULTS / 'summary.json').read_text())
for name, digest in summary['files_sha256'].items():
    assert sha256(ROOT / name) == digest, f'Pilot artifact changed: {name}'
for name, digest in summary['implementation_sha256'].items():
    assert sha256(PROJECT / name) == digest, f'Pilot implementation changed: {name}'
if VERIFY_DOME:
    display(verify(ROOT))
display(pd.Series({k: summary[k] for k in [
    'planned_start','planned_end','replayed_start','replayed_end','full_week_complete',
    'events','tokens','baskets','messages','fully_observed_event_hours','event_hours',
    'conditional_positive_event_hours','maximum_conditional_one_basket_gross_usd',
    'historical_fees_verified','net_profit_estimated']}))

{'unchanged': True, 'files': 2913}

planned_start                               2026-05-04 01:00:00+00:00
planned_end                                 2026-05-11 01:00:00+00:00
replayed_start                              2026-05-04 01:00:00+00:00
replayed_end                                2026-05-04 02:00:00+00:00
full_week_complete                                              False
events                                                             15
tokens                                                            200
baskets                                                           301
messages                                                       435971
fully_observed_event_hours                                          2
event_hours                                                        15
conditional_positive_event_hours                                    0
maximum_conditional_one_basket_gross_usd                          0.0
historical_fees_verified                                        False
net_profit_estimated

In [3]:
display(pd.read_parquet(ROOT / 'events.parquet'))
hours = pd.read_csv(RESULTS / 'event_hours.csv')
display(hours.groupby('classification').agg(
    event_hours=('event_id','size'),
    complete_seconds=('fully_observed_seconds','sum'),
    maximum_conditional_gross_usd=('conditional_peak_gross_usd','max')))
display(hours[['event_slug','hour','usable_token_fraction','fully_observed_seconds',
               'quoted_gap','conditional_peak_gross_usd','fee_net_executable_status']])

,event_id,event_slug,classification,sector,historical_metadata_verified,current_inventory_members
0,48297,databricks-ipo-closing-market-cap,negrisk,business,False,8
1,106981,bitcoin-vs-gold-vs-sp-500-in-2026,negrisk,crypto,False,3
2,230330,us-trade-deficit-in-2026,negrisk,macro,False,8
3,228259,how-many-tornadoes-in-the-us-in-2026,negrisk,other,False,8
4,401356,colombia-presidential-election-1st-round-turnout,negrisk,politics,False,6
5,417522,chi1-edv-ccu-2026-05-22,negrisk,sports,False,3
6,200252,openai-ipo-closing-market-cap-554,negrisk,business,False,7
7,201560,india-annual-inflation-2026,negrisk,macro,False,7
8,439906,zelenskyy-of-tweets-may-5-may-12-2026,negrisk,other,False,11
9,204610,of-seats-won-by-ppp-in-south-korea-by-elections,negrisk,politics,False,7


,event_hours,complete_seconds,maximum_conditional_gross_usd
classification,,,
ladder,5,10046.357,0.0
negrisk,10,29626.921,0.0


,event_slug,hour,usable_token_fraction,fully_observed_seconds,quoted_gap,conditional_peak_gross_usd,fee_net_executable_status
0,databricks-ipo-closing-market-cap,2026-05-04 01:00:00+00:00,1.000000,2943.019,-0.100,0.0,unknown_historical_fees_and_metadata
1,bitcoin-vs-gold-vs-sp-500-in-2026,2026-05-04 01:00:00+00:00,1.000000,3600.000,-0.050,0.0,unknown_historical_fees_and_metadata
2,us-trade-deficit-in-2026,2026-05-04 01:00:00+00:00,0.875000,2534.877,NaN,0.0,unknown_historical_fees_and_metadata
3,how-many-tornadoes-in-the-us-in-2026,2026-05-04 01:00:00+00:00,1.000000,3348.323,-0.109,0.0,unknown_historical_fees_and_metadata
4,colombia-presidential-election-1st-round-turnout,2026-05-04 01:00:00+00:00,1.000000,3433.381,-1.030,0.0,unknown_historical_fees_and_metadata
5,chi1-edv-ccu-2026-05-22,2026-05-04 01:00:00+00:00,1.000000,3600.000,-0.990,0.0,unknown_historical_fees_and_metadata
6,openai-ipo-closing-market-cap-554,2026-05-04 01:00:00+00:00,1.000000,3370.969,-1.326,0.0,unknown_historical_fees_and_metadata
7,india-annual-inflation-2026,2026-05-04 01:00:00+00:00,1.000000,3131.502,-0.491,0.0,unknown_historical_fees_and_metadata
8,zelenskyy-of-tweets-may-5-may-12-2026,2026-05-04 01:00:00+00:00,0.636364,1047.445,NaN,0.0,unknown_historical_fees_and_metadata
9,of-seats-won-by-ppp-in-south-korea-by-elections,2026-05-04 01:00:00+00:00,1.000000,2617.405,-0.202,0.0,unknown_historical_fees_and_metadata


In [4]:
quality = pd.read_csv(RESULTS / 'token_quality.csv')
display(quality)
display(pd.Series({k: summary[k] for k in ['transfer_bytes','raw_cache_bytes','assumptions','quality_limits']}))

,asset_id,usable_at_end,source_baseline_ms,receive_ms,price_change,older_revisions_ignored,book,rebased_levels,equal_source_rebased_levels,top_mismatch_batches,ambiguous_source_best_batches,last_trade_price,older_snapshots_ignored
0,1013303232772613078089364769282479364842719188...,False,1777858818284,1777859996485,2279,96.0,61,52,42,123.0,1.0,NaN,NaN
1,1025205157286985031646867846747978044328792260...,True,1777859865441,1777859996108,1855,105.0,67,50,45,63.0,NaN,NaN,NaN
2,1028431895329686584841247243841691454310968435...,True,1777859869869,1777859997451,1725,20.0,46,11,6,NaN,NaN,NaN,NaN
3,1030857673646428597599568112983351459583344748...,False,1777859468720,1777859998468,2848,196.0,78,60,42,285.0,NaN,3.0,NaN
4,1031029027005737910334429611245959257202457300...,True,1777859492991,1777859996904,464,10.0,92,84,79,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,9779779365142063138759332447581984349844925758...,True,1777859872072,1777859976333,349,2.0,78,74,72,NaN,NaN,NaN,NaN
196,9841071128013801426343304957794320306607184788...,True,1777858989313,1777859999180,4363,182.0,44,28,19,69.0,NaN,NaN,NaN
197,9865378754846136456030025275933434374337253405...,True,1777859504005,1777859943859,202,3.0,104,93,93,NaN,NaN,NaN,NaN
198,9943734038648994121075062636841914198681847773...,True,1777859772981,1777859986718,1671,175.0,68,57,43,NaN,NaN,NaN,NaN


transfer_bytes                                             718610368
raw_cache_bytes                                              4885513
assumptions        [current rules/membership retrospectively fixe...
quality_limits     [best-quote consistency is necessary, not proo...
dtype: object

**Interpretation:** quoted margins are conditional zero-fee diagnostics with an assumed initial tick and minimum size. They are not estimated collectible dollars. Historical fees/rules, recording continuity, subscription completeness and 100 ms fills still require separate validation. A missing or unusable book produces an unknown label, not a negative example. Repeated quotes and overlapping pairs are never summed as profit.

Continue acquisition explicitly with `python -m src.data.pmxt_pilot download --hours 169 --transfer-gb <budget>` after inspecting the measured transfer/storage needs. `audit` replays only the contiguous cached prefix and never claims the missing planned hours were observed. Expansion should use a newly frozen development/test design; no PMXT model is selected here.